<a href="https://colab.research.google.com/github/Jayalakshmi-27-05/Syntecxhub_projects/blob/main/2_Syntecxhub_Sentiment_Analysis_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install gradio pandas scikit-learn gTTS openai-whisper

In [3]:
import gradio as gr
import pandas as pd
import re
import sqlite3
import time
import whisper

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from gtts import gTTS

In [4]:
df = pd.read_csv("/content/IMDB Dataset.csv", engine='python', on_bad_lines='skip', encoding='latin1')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['cleaned'] = df['review'].apply(clean_text)
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

In [7]:
df.dropna(subset=['sentiment'], inplace=True)
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['cleaned'])
y = df['sentiment']

model = LogisticRegression(max_iter=200)
model.fit(X, y)

LogisticRegression(max_iter=200)

In [8]:
!pip install deep_translator
from deep_translator import GoogleTranslator

def translate_to_english(text, lang):
    if lang != "English":
        return GoogleTranslator(source='auto', target='en').translate(text)
    return text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.0 MB/s eta 0:00:00


In [9]:
whisper_model = whisper.load_model("base")

def voice_to_text(audio):
    try:
        result = whisper_model.transcribe(audio)
        return result["text"]
    except:
        return ""

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 84.8MiB/s]


In [10]:
def text_to_speech(text):
    tts = gTTS(text=text)
    file = "response.mp3"
    tts.save(file)
    return file

In [11]:
def predict_sentiment(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])
    prob = model.predict_proba(vec)[0]

    sentiment = "Positive 😊" if prob[1] > prob[0] else "Negative 😞"
    return sentiment

In [12]:
def chatbot_response(message, history, lang):

    if not message:
        return "", history, None

    message = translate_to_english(message, lang)

    # typing animation
    history.append((message, "Typing..."))
    yield "", history, None

    time.sleep(1)

    result = predict_sentiment(message)
    history[-1] = (message, result)

    audio = text_to_speech(result)

    yield "", history, audio

In [13]:
conn = sqlite3.connect("users.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    username TEXT,
    password TEXT
)
""")

conn.commit()
conn.close()

In [14]:
def signup(user, pwd):
    conn = sqlite3.connect("users.db")
    cur = conn.cursor()
    cur.execute("INSERT INTO users VALUES (?, ?)", (user, pwd))
    conn.commit()
    conn.close()
    return "Signup successful ✅"

def login(user, pwd):
    conn = sqlite3.connect("users.db")
    cur = conn.cursor()
    cur.execute("SELECT * FROM users WHERE username=? AND password=?", (user, pwd))
    result = cur.fetchone()
    conn.close()

    return "Login successful ✅" if result else "Invalid ❌"

In [15]:
with gr.Blocks(
    theme=gr.themes.Base(),
    css="body {background:#343541;} .chatbot {height:500px;}"
) as app:

    with gr.Tab("🔐 Login"):
        user = gr.Textbox(label="Username")
        pwd = gr.Textbox(label="Password", type="password")

        login_btn = gr.Button("Login")
        signup_btn = gr.Button("Signup")
        out = gr.Textbox()

        login_btn.click(login, [user, pwd], out)
        signup_btn.click(signup, [user, pwd], out)

    with gr.Tab("💬 Chatbot"):

        chatbot = gr.Chatbot()
        audio_out = gr.Audio(label="AI Voice")

        msg = gr.Textbox(placeholder="Send a message...")
        lang = gr.Dropdown(["English","Hindi","Telugu"], value="English")

        send = gr.Button("Send")

        audio_in = gr.Audio(sources=["microphone"], type="filepath")

        send.click(chatbot_response, [msg, chatbot, lang], [msg, chatbot, audio_out])

        msg.submit(chatbot_response, [msg, chatbot, lang], [msg, chatbot, audio_out])

        audio_in.change(voice_to_text, audio_in, msg).then(
            chatbot_response,
            [msg, chatbot, lang],
            [msg, chatbot, audio_out]
        )

app.launch()

/tmp/ipykernel_5890/1179196272.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_5890/1179196272.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_5890/1179196272.py:19: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_5890/1179196272.py:19: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you 

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b510db812f230b216c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


https://b510db812f230b216c.gradio.live/

this is my gradio link
